# DBSCAN lab: clusters without choosing k
Explore density-based clustering on a small planar point set. Change eps and min_samples to see how clusters and noise change.

Coordinates are synthetic projected units. DBSCAN's distance metric and parameter scale must be meaningful for real data.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'implementations').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'implementations/python/dbscan'))
from dbscan import dbscan


## 1. Inspect the points

The sample has three dense groups, a bridge-like set of points and isolated observations. The cluster structure is not supplied to the algorithm.

In [ ]:
rng = np.random.default_rng(23)
a = rng.normal(loc=(0, 0), scale=(0.22, 0.22), size=(35, 2))
b = rng.normal(loc=(2, 0), scale=(0.22, 0.22), size=(35, 2))
c = rng.normal(loc=(1, 1.8), scale=(0.22, 0.22), size=(35, 2))
noise = np.array([[3.5, 3.2], [-1.8, 2.8], [3.6, -1.4]])
points = np.vstack([a, b, c, noise])
fig, ax = plt.subplots(figsize=(7, 6)); ax.scatter(points[:, 0], points[:, 1], s=25)
ax.set_title('Synthetic point pattern'); ax.set_aspect('equal'); ax.grid(True); plt.show()


## 2. Cluster by density

A core point has at least min_samples points in its eps-neighbourhood, including itself. Border points join a cluster through a core point; points outside all density-connected clusters are noise.

In [ ]:
eps = 0.55
min_samples = 5
labels = np.asarray(dbscan(points, eps=eps, min_samples=min_samples))
fig, ax = plt.subplots(figsize=(7, 6))
for label in sorted(set(labels)):
    selected = labels == label
    ax.scatter(points[selected, 0], points[selected, 1], s=30, label='Noise' if label == -1 else f'Cluster {label}')
ax.set_title(f'DBSCAN: eps={eps}, min_samples={min_samples}'); ax.set_aspect('equal'); ax.legend(); ax.grid(True); plt.show()
print('Cluster labels and counts:', {int(label): int(np.sum(labels == label)) for label in sorted(set(labels))})


## 3. Change the parameters

Try a smaller and a larger eps. A larger radius can merge nearby groups; a smaller radius can turn border points into noise. Raising min_samples makes core status harder to achieve.

In [ ]:
for radius, minimum in [(0.35, 5), (0.75, 5), (0.55, 9)]:
    result = dbscan(points, eps=radius, min_samples=minimum)
    clusters = set(result) - {-1}
    print(f'eps={radius:.2f}, min_samples={minimum}: {len(clusters)} clusters, {result.count(-1)} noise points')


## 4. Investigate

1. Why does DBSCAN not need the number of clusters in advance?
2. What happens when eps is too small or too large?
3. How does the result change when sampling density varies across the study area?
4. Why can a border point be assigned according to traversal order?
5. What spatial index could replace the pairwise distance matrix for larger data?